# Ollama 연결·도구 호출 테스트

VM의 Ollama 서버에 노트북에서 붙어서, 후보 모델(qwen3.5:9b, gemma4:12b)이 **도구 호출**을 제대로 하는지 비교합니다.

**실행 전 준비**
1. VM: tmux 안에서 `OLLAMA_KEEP_ALIVE=1h OLLAMA_CONTEXT_LENGTH=8192 ollama serve` 실행 중
2. VM: `ollama pull qwen3.5:9b`, `ollama pull gemma4:12b` 완료
3. 노트북 터미널: SSH 터널 `ssh -L 11434:localhost:11434 ...` 창 열어 두기
4. 레포 폴더에서 `uv add langchain-ollama` 후 커널로 `.venv` 선택

In [1]:
import time
import requests

BASE_URL = "http://localhost:11434"
MODELS = ["qwen3.5:9b", "gemma4:12b"]
NUM_CTX = 8192

## 1. 서버 연결 확인

In [2]:
print(requests.get(BASE_URL).text)  # "Ollama is running" 이 나오면 OK

tags = requests.get(f"{BASE_URL}/api/tags").json()
print("받아 둔 모델:", [m["name"] for m in tags["models"]])

Ollama is running
받아 둔 모델: ['gemma4:12b', 'qwen3.5:9b']


## 2. 일반 답변 + 속도

In [5]:
from langchain_ollama import ChatOllama

for model in MODELS:
    llm = ChatOllama(base_url=BASE_URL, model=model, num_ctx=NUM_CTX, temperature=0)
    start = time.time()
    res = llm.invoke("RFP(제안요청서)가 뭔지 한국어로 두 문장으로 설명해줘.")
    print(f"=== {model} ({time.time() - start:.1f}초) ===")
    print(res.content, "\n")

=== qwen3.5:9b (23.2초) ===
제안요청서 (RFP) 는 구매자가 특정 프로젝트나 서비스를 위해 잠재적인 공급업체들에게 요구 사항과 평가 기준을 명시한 공식 문서입니다. 이를 통해 업체들은 자신의 솔루션을 제안하고, 구매자는 그 중 가장 적합한 파트너를 선정하여 계약을 체결합니다. 

=== gemma4:12b (9.3초) ===
RFP는 기업이나 기관이 특정 프로젝트나 서비스에 필요한 요구사항을 정리하여 업체들에게 제안을 요청하는 문서입니다. 이를 통해 업체들은 구체적인 실행 계획과 비용 등을 포함한 제안서를 제출하며, 발주처는 이를 비교하여 최적의 파트너를 선정하게 됩니다. 



## 3. GPU에 올라갔는지 확인
`size_vram` 이 `size` 와 같으면 전부 GPU에 올라간 것 (VM에서 `ollama ps` 로 봐도 같음)

In [6]:
for m in requests.get(f"{BASE_URL}/api/ps").json()["models"]:
    print(m["name"], f"전체 {m['size']/1e9:.1f}GB", f"GPU {m['size_vram']/1e9:.1f}GB")

gemma4:12b 전체 1.5GB GPU 1.5GB
qwen3.5:9b 전체 5.9GB GPU 5.9GB


## 4. 도구 호출 테스트
에이전트에 쓰려면 질문을 받고 **도구를 골라 인자를 채워서 호출**할 수 있어야 합니다.
`tool_calls` 가 비어 있으면 그 모델은 에이전트용으로 부적합.

In [7]:
from langchain_core.tools import tool


@tool
def search_rfp(query: str) -> str:
    """RFP 문서 본문에서 질문과 관련된 내용을 검색합니다."""
    return "[OO시청_통합관제시스템_구축.hwp] 사업 예산은 5억 원(부가세 포함)이며, 사업 기간은 계약일로부터 6개월이다."


@tool
def filter_rfp_by_metadata(agency: str = "", min_budget: int = 0) -> str:
    """발주기관, 최소 사업금액(원) 같은 조건으로 공고 목록을 찾습니다."""
    return "OO시청_통합관제시스템_구축 (5억 원), OO시청_홈페이지_개편 (2억 원)"


TOOLS = [search_rfp, filter_rfp_by_metadata]
QUESTIONS = [
    "OO시청 통합관제시스템 사업 예산이 얼마야?",   # → search_rfp 기대
    "OO시청에서 낸 3억 원 이상 공고 찾아줘",        # → filter_rfp_by_metadata 기대
    "안녕? 너는 뭘 할 수 있어?",                    # → 도구 호출 없이 답하는 게 정상
]

for model in MODELS:
    llm = ChatOllama(base_url=BASE_URL, model=model, num_ctx=NUM_CTX, temperature=0).bind_tools(TOOLS)
    print(f"=== {model} ===")
    for q in QUESTIONS:
        start = time.time()
        res = llm.invoke(q)
        calls = [(c["name"], c["args"]) for c in res.tool_calls]
        print(f"- {q}\n  → {calls or '도구 호출 없음'} ({time.time() - start:.1f}초)")
    print()

=== qwen3.5:9b ===
- OO시청 통합관제시스템 사업 예산이 얼마야?
  → [('search_rfp', {'query': 'OO시청 통합관제시스템 사업 예산'})] (10.0초)
- OO시청에서 낸 3억 원 이상 공고 찾아줘
  → [('filter_rfp_by_metadata', {'agency': 'OO시청', 'min_budget': 300000000})] (5.0초)
- 안녕? 너는 뭘 할 수 있어?
  → 도구 호출 없음 (7.0초)

=== gemma4:12b ===
- OO시청 통합관제시스템 사업 예산이 얼마야?
  → [('search_rfp', {'query': 'OO시청 통합관제시스템'})] (2.2초)
- OO시청에서 낸 3억 원 이상 공고 찾아줘
  → [('filter_rfp_by_metadata', {'agency': 'OO시청', 'min_budget': 300000000})] (3.2초)
- 안녕? 너는 뭘 할 수 있어?
  → 도구 호출 없음 (17.2초)



## 5. 도구 결과까지 넣고 최종 답변
도구 결과를 받은 뒤 **근거를 인용해서** 답하는지, 없는 내용을 지어내지 않는지 확인

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

SYSTEM = "너는 공공 입찰 RFP 분석 어시스턴트야. 반드시 도구 결과에 있는 내용만 근거로 답하고, 출처 파일명을 함께 적어. 모르면 모른다고 해."
question = "OO시청 통합관제시스템 사업 예산이랑 사업 기간 알려줘. 입찰 마감일도."

for model in MODELS:
    llm = ChatOllama(base_url=BASE_URL, model=model, num_ctx=NUM_CTX, temperature=0).bind_tools(TOOLS)
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    ai = llm.invoke(messages)
    messages.append(ai)
    for call in ai.tool_calls:
        result = {t.name: t for t in TOOLS}[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(result, tool_call_id=call["id"]))
    final = llm.invoke(messages)
    print(f"=== {model} ===\n도구: {[c['name'] for c in ai.tool_calls]}\n{final.content}\n")
# 마감일은 도구 결과에 없으므로 "확인되지 않는다"고 답해야 정상

=== qwen3.5:9b ===
도구: ['search_rfp']
OO시청 통합관제시스템 사업의 예산은 5억 원(부가세 포함)이며, 사업 기간은 계약일로부터 6개월입니다. (출처: OO시청_통합관제시스템_구축.hwp)

입찰 마감일에 대한 정보는 제공된 문서에서 확인되지 않았습니다.

=== gemma4:12b ===
도구: ['search_rfp']
OO시청 통합관제시스템 사업의 예산, 사업 기간, 입찰 마감일 정보는 다음과 같습니다.

*   **사업 예산:** 5억 원(부가세 포함)
*   **사업 기간:** 계약일로부터 6개월
*   **입찰 마감일:** 해당 정보는 제공된 문서에 포함되어 있지 않습니다.

출처: OO시청_통합관제시스템_구축.hwp



## 결과 정리 (직접 채우기)

| 항목 | qwen3.5:9b | gemma4:12b |
| --- | --- | --- |
| 일반 답변 속도 | 23.2s | 9.3s |
| GPU 100% 로드 | 5.9GB GPU 100% | 1.5GB GPU 100% ⚠️ |
| 도구 선택 정확 (3문항) | 3/3 정확 | 3/3 정확 |
| 근거 인용 / 마감일 지어내지 않음 | 출처 표시 ✓, 마감일은 "확인되지 않음" ✓ | 출처 표시 ✓, 마감일은 "포함되어 있지 않음" ✓ |
| 한국어 품질 | 자연스러움, 문장형 | 자연스러움, 목록형으로 정리 |

처음 실행 시 모델 로딩때문에 더 오래 걸릴 수 있음. 
⚠️ gemma의 파일 크기는 원래 8GB인데 표시가 잘못 나옴. VM에서 확인 시 제대로 동작 중.